In [28]:
import os

# Define the directory structure
directories = [
    'braintumnet/configs',
    'braintumnet/data/raw',
    'braintumnet/data/processed',
    'braintumnet/src/braintumnet/data',
    'braintumnet/src/braintumnet/models',
    'braintumnet/src/braintumnet/engine',
    'braintumnet/src/braintumnet/utils',
    'braintumnet/scripts',
    'braintumnet/tests'
]

# Create directories
for directory in directories:
    os.makedirs(directory, exist_ok=True)

# Create __init__.py files
init_files = [
    'braintumnet/src/braintumnet/__init__.py',
    'braintumnet/src/braintumnet/data/__init__.py',
    'braintumnet/src/braintumnet/models/__init__.py',
    'braintumnet/src/braintumnet/engine/__init__.py',
    'braintumnet/src/braintumnet/utils/__init__.py'
]

for init_file in init_files:
    with open(init_file, 'w') as f:
        f.write('# This file makes the directory a Python package\n')

print("Project folders and __init__.py files created successfully!")

# Verify the structure
print("\nDirectory structure:")
for root, dirs, files in os.walk('braintumnet'):
    level = root.replace('braintumnet', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

Project folders and __init__.py files created successfully!

Directory structure:
braintumnet/
  requirements.txt
  configs/
    default.yaml
  data/
    processed/
    raw/
  scripts/
    evaluate.py
    prepare_brats2020.py
    train.py
    visualize_batch.py
  src/
    braintumnet/
      losses/base.py
      metrics/base.py
      __init__.py
      data/
        brats2020_dataset.py
        preprocessing.py
        transforms.py
        __init__.py
      engine/
        evaluator.py
        trainer.py
        __init__.py
      models/
        braintumnet.py
        cbam.py
        masked_transformer.py
        seg_unet.py
        t_inception.py
        __init__.py
      utils/
        io.py
        seed.py
        __init__.py
  tests/


In [29]:
%%writefile braintumnet/requirements.txt
torch>=2.1
torchvision>=0.16
numpy>=1.23
pillow>=10.0
pyyaml>=6.0
nibabel>=5.2
scikit-image>=0.22
scikit-learn>=1.3
scipy>=1.11
tqdm>=4.66


Overwriting braintumnet/requirements.txt


In [30]:
%%writefile braintumnet/configs/default.yaml
exp_name: "braintumnet_brats2020"

data:
  raw_root: "braintumnet/data/raw"
  proc_root: "braintumnet/data/processed"
  modality: "t1ce"         # one of: t1, t1ce, t2, flair, multi(=4ch)
  img_size: 256
  slices_per_case: 20
  tumor_slice_ratio: 0.7    # fraction of tumor slices in dataset
  num_folds: 5
  fold: 0

train:
  epochs: 250
  batch_size: 16
  lr: 1.0e-4
  weight_decay: 0.0
  workers: 4
  seg_loss_weight: 1.0
  cls_loss_weight: 0.7
  scheduler: "cosine"       # cosine or none
  amp: true                 # mixed precision

model:
  in_channels: 1            # 1 for single modality, 4 for multi
  num_classes_seg: 1
  num_classes_cls: 2        # HGG vs LGG
  base: 32
  patch_size: 8
  dim: 256
  n_heads: 4
  depth: 2
  roi_stop_grad: true

augment:
  rotate_deg: 30
  hflip_p: 0.5
  vflip_p: 0.5

logging:
  out_dir: "runs"
  save_dir: "checkpoints"


Overwriting braintumnet/configs/default.yaml


In [31]:
%%writefile braintumnet/src/braintumnet/utils/seed.py
import os, random, numpy as np, torch

def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


Overwriting braintumnet/src/braintumnet/utils/seed.py


In [32]:
%%writefile braintumnet/src/braintumnet/utils/io.py
import os, yaml, torch
from typing import Any, Dict

def load_yaml(path: str) -> Dict[str, Any]:
    with open(path, "r") as f:
        return yaml.safe_load(f)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def save_ckpt(model, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(model.state_dict(), path)

def load_ckpt(model, path: str, map_location="cpu"):
    sd = torch.load(path, map_location=map_location)
    model.load_state_dict(sd)
    return model


Overwriting braintumnet/src/braintumnet/utils/io.py


In [33]:
%%writefile braintumnet/src/braintumnet/data/transforms.py
import random
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF
import torch

def resize_pad_to_square(img: Image.Image, size: int, is_mask: bool=False) -> Image.Image:
    w, h = img.size
    s = max(w, h)
    pad_l = (s - w) // 2
    pad_t = (s - h) // 2
    pad_r = s - w - pad_l
    pad_b = s - h - pad_t
    fill = 0
    if is_mask:
        img = TF.pad(img, [pad_l, pad_t, pad_r, pad_b], fill=0)
        img = img.resize((size, size), Image.NEAREST)
    else:
        img = TF.pad(img, [pad_l, pad_t, pad_r, pad_b], fill=0)
        img = img.resize((size, size), Image.BILINEAR)
    return img

def to_tensor01(img: Image.Image) -> torch.Tensor:
    arr = np.asarray(img).astype(np.float32)
    if arr.max() > 1.0: arr /= 255.0
    return torch.from_numpy(arr).unsqueeze(0)  # (1,H,W)

def augment_pair(img: Image.Image, msk: Image.Image, img_size: int,
                 rotate_deg: int=30, hflip_p: float=0.5, vflip_p: float=0.5,
                 train: bool=True):
    img = resize_pad_to_square(img, img_size, is_mask=False)
    msk = resize_pad_to_square(msk, img_size, is_mask=True)
    if train:
        angle = random.uniform(-rotate_deg, rotate_deg)
        img = TF.rotate(img, angle)
        msk = TF.rotate(msk, angle)
        if random.random() < hflip_p:
            img = TF.hflip(img); msk = TF.hflip(msk)
        if random.random() < vflip_p:
            img = TF.vflip(img); msk = TF.vflip(msk)
    return to_tensor01(img), (torch.from_numpy((np.asarray(msk)>127).astype(np.float32)).unsqueeze(0))


Overwriting braintumnet/src/braintumnet/data/transforms.py


In [34]:
%%writefile braintumnet/src/braintumnet/data/brats2020_dataset.py
import os
from typing import List, Dict
from PIL import Image
import torch
from torch.utils.data import Dataset
from .transforms import augment_pair

class SliceDataset(Dataset):
    """
    processed/
      images/<slice_id>.png    (grayscale or 4ch .npy if multi)
      masks/<slice_id>.png     (0/255)
      labels.csv               (case_id,label)
      mapping.csv              (slice_id,case_id)
      split_train_fold{k}.txt
      split_val_fold{k}.txt
    """
    def __init__(self, proc_root: str, split_file: str,
                 img_size: int=256, rotate_deg: int=30, hflip_p: float=0.5, vflip_p: float=0.5,
                 train: bool=True, in_channels: int=1):
        self.proc_root = proc_root
        self.train = train
        self.img_size = img_size
        self.rotate_deg, self.hflip_p, self.vflip_p = rotate_deg, hflip_p, vflip_p
        self.in_channels = in_channels
        with open(split_file, "r") as f:
            self.slice_ids: List[str] = [x.strip() for x in f if x.strip()]

        # labels
        self.case_label: Dict[str, int] = {}
        labels_csv = os.path.join(proc_root, "labels.csv")
        if os.path.exists(labels_csv):
            with open(labels_csv) as f:
                for line in f:
                    if "," in line:
                        cid, lab = line.strip().split(",")
                        self.case_label[cid] = int(lab)
        # mapping slice -> case
        self.slice_case: Dict[str, str] = {}
        mapping_csv = os.path.join(proc_root, "mapping.csv")
        if os.path.exists(mapping_csv):
            with open(mapping_csv) as f:
                for line in f:
                    if "," in line:
                        sid, cid = line.strip().split(",")
                        self.slice_case[sid] = cid

    def __len__(self): return len(self.slice_ids)

    def _load_image(self, sid: str) -> Image.Image:
        img_path = os.path.join(self.proc_root, "images", f"{sid}.png")
        if not os.path.exists(img_path):
            raise FileNotFoundError(img_path)
        return Image.open(img_path).convert("L")

    def _load_mask(self, sid: str) -> Image.Image:
        msk_path = os.path.join(self.proc_root, "masks", f"{sid}.png")
        if not os.path.exists(msk_path):
            raise FileNotFoundError(msk_path)
        return Image.open(msk_path).convert("L")

    def __getitem__(self, idx):
        sid = self.slice_ids[idx]
        img = self._load_image(sid)
        msk = self._load_mask(sid)
        img_t, msk_t = augment_pair(img, msk, self.img_size, self.rotate_deg, self.hflip_p, self.vflip_p, self.train)
        cid = self.slice_case.get(sid, sid.split("_")[0])
        label = self.case_label.get(cid, 0)
        return {"image": img_t, "mask": msk_t, "label": torch.tensor(label, dtype=torch.long), "slice_id": sid, "case_id": cid}


Overwriting braintumnet/src/braintumnet/data/brats2020_dataset.py


In [35]:
%%writefile braintumnet/src/braintumnet/data/preprocessing.py
import os, glob, csv
from typing import Tuple, List
import numpy as np
from PIL import Image
import nibabel as nib

def _rescale01(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    nz = arr > 0
    if nz.sum() > 0:
        a = arr[nz]
        lo, hi = a.min(), a.max()
    else:
        lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-6:
        return np.zeros_like(arr, dtype=np.float32)
    out = (arr - lo) / (hi - lo)
    out[~np.isfinite(out)] = 0
    return out

def _save_png01(x: np.ndarray, path: str):
    x = (x * 255.0).clip(0,255).astype(np.uint8)
    Image.fromarray(x).save(path)

def _find_modal_paths(case_dir: str, modality: str) -> Tuple[str, str]:
    # modality file names contain: t1, t1ce, t2, flair
    patt = f"*{modality}*.nii*"
    img = glob.glob(os.path.join(case_dir, patt))
    seg = glob.glob(os.path.join(case_dir, "*seg*.nii*"))
    if len(img)==0 or len(seg)==0:
        return None, None
    return img[0], seg[0]

def _pick_slices(seg3d: np.ndarray, k: int) -> List[int]:
    zs = np.where(seg3d.sum(axis=(0,1)) > 0)[0]
    if len(zs) == 0:
        return []
    if len(zs) <= k:
        return zs.tolist()
    idx = np.linspace(zs[0], zs[-1], k).astype(int).tolist()
    return idx

def process_brats2020(raw_root: str, out_root: str, modality: str="t1ce",
                      img_size: int=256, slices_per_case: int=20, tumor_slice_ratio: float=0.7):
    os.makedirs(os.path.join(out_root, "images"), exist_ok=True)
    os.makedirs(os.path.join(out_root, "masks"), exist_ok=True)
    labels_path = os.path.join(out_root, "labels.csv")
    mapping_path = os.path.join(out_root, "mapping.csv")

    cases = []
    for grp, lab in [("HGG", 0), ("LGG", 1)]:
        grp_dir = os.path.join(raw_root, grp)
        if not os.path.isdir(grp_dir): continue
        for case_dir in sorted(glob.glob(os.path.join(grp_dir, "*"))):
            cases.append((case_dir, lab))

    with open(labels_path, "w", newline="") as lf, open(mapping_path, "w", newline="") as mf:
        lw, mw = csv.writer(lf), csv.writer(mf)
        lw.writerow(["case_id","label"])
        mw.writerow(["slice_id","case_id"])
        total_slices = 0

        for case_dir, lab in cases:
            case_id = os.path.basename(case_dir)
            lw.writerow([case_id, lab])

            img_path, seg_path = _find_modal_paths(case_dir, modality)
            if img_path is None: 
                print("Skip (missing files):", case_dir)
                continue

            img3d = nib.load(img_path).get_fdata()
            seg3d = nib.load(seg_path).get_fdata()
            img3d = _rescale01(img3d)
            wt = (seg3d > 0).astype(np.float32)

            tumor_z = np.where(wt.sum(axis=(0,1)) > 0)[0].tolist()
            non_z   = [z for z in range(img3d.shape[2]) if z not in set(tumor_z)]

            k_tum = min(len(tumor_z), int(round(slices_per_case * tumor_slice_ratio)))
            k_non = slices_per_case - k_tum
            pick_t = np.linspace(tumor_z[0], tumor_z[-1], k_tum).astype(int).tolist() if len(tumor_z)>0 and k_tum>0 else []
            if k_non > 0 and len(non_z)>0:
                # sample uniformly spread
                step = max(1, len(non_z)//k_non)
                pick_n = non_z[::step][:k_non]
            else:
                pick_n = []
            picks = sorted(set(pick_t + pick_n))

            for z in picks:
                img = img3d[:,:,z]
                msk = wt[:,:,z]
                # pad to square then resize
                h, w = img.shape
                s = max(h,w)
                pad_h = (s - h); pad_w = (s - w)
                img_p = np.pad(img, ((pad_h//2, pad_h - pad_h//2), (pad_w//2, pad_w - pad_w//2)), mode="constant")
                msk_p = np.pad(msk, ((pad_h//2, pad_h - pad_h//2), (pad_w//2, pad_w - pad_w//2)), mode="constant")
                img_p = Image.fromarray((img_p*255).astype(np.uint8)).resize((img_size,img_size), Image.BILINEAR)
                msk_p = Image.fromarray((msk_p*255).astype(np.uint8)).resize((img_size,img_size), Image.NEAREST)
                sid = f"{case_id}_{int(z):03d}"
                _save_png01(np.array(img_p).astype(np.float32)/255.0, os.path.join(out_root, "images", f"{sid}.png"))
                Image.fromarray(np.array(msk_p)).save(os.path.join(out_root, "masks", f"{sid}.png"))
                mw.writerow([sid, case_id])
                total_slices += 1
        print("Processed slices:", total_slices)

def make_folds(proc_root: str, num_folds: int=5):
    import csv, random
    labels_csv = os.path.join(proc_root, "labels.csv")
    mapping_csv = os.path.join(proc_root, "mapping.csv")
    assert os.path.exists(labels_csv) and os.path.exists(mapping_csv), "Run processing first."

    # case -> label
    case_label = {}
    with open(labels_csv) as f:
        r = csv.DictReader(f)
        for row in r:
            case_label[row["case_id"]] = int(row["label"])
    # case -> slice_ids
    case_slices = {}
    with open(mapping_csv) as f:
        r = csv.DictReader(f)
        for row in r:
            case_slices.setdefault(row["case_id"], []).append(row["slice_id"])

    # stratified split on cases
    cases0 = [c for c,l in case_label.items() if l==0]
    cases1 = [c for c,l in case_label.items() if l==1]
    random.seed(42)
    random.shuffle(cases0); random.shuffle(cases1)
    folds = [[] for _ in range(num_folds)]
    for i,c in enumerate(cases0): folds[i%num_folds].append(c)
    for i,c in enumerate(cases1): folds[i%num_folds].append(c)

    # write slice ids per fold
    for k in range(num_folds):
        val_cases = set(folds[k])
        tr_cases = set(case_label.keys()) - val_cases
        tr_slices, val_slices = [], []
        for cid in tr_cases: tr_slices += case_slices[cid]
        for cid in val_cases: val_slices += case_slices[cid]
        with open(os.path.join(proc_root, f"split_train_fold{k}.txt"), "w") as f: f.write("\n".join(tr_slices))
        with open(os.path.join(proc_root, f"split_val_fold{k}.txt"), "w") as f: f.write("\n".join(val_slices))
    print("Folds written:", num_folds)


Overwriting braintumnet/src/braintumnet/data/preprocessing.py


In [36]:
%%writefile braintumnet/src/braintumnet/models/cbam.py
import torch, torch.nn as nn, torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels//reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels//reduction, in_channels, 1, bias=False),
        )
    def forward(self, x):
        att = torch.sigmoid(self.mlp(self.avg(x)) + self.mlp(self.max(x)))
        return x * att

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
    def forward(self, x):
        att = torch.cat([x.mean(1, True), x.amax(1, True)], dim=1)
        att = torch.sigmoid(self.conv(att))
        return x * att

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(in_channels, reduction)
        self.sa = SpatialAttention(k)
    def forward(self, x):
        return self.sa(self.ca(x))


Overwriting braintumnet/src/braintumnet/models/cbam.py


In [37]:
%%writefile braintumnet/src/braintumnet/models/masked_transformer.py
import torch, torch.nn as nn, torch.nn.functional as F

class PatchEmbed(nn.Module):
    def __init__(self, in_ch, embed_dim, patch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch, stride=patch)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x)  # B,C,H',W'
        B,C,H,W = x.shape
        x = x.flatten(2).transpose(1,2)  # B,N,C
        x = self.norm(x)
        return x, (H,W)

class SoftMaskGenerator(nn.Module):
    def __init__(self, dim, hidden=128, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(),
            nn.Linear(hidden, n_heads), nn.Sigmoid()
        )
    def forward(self, tokens):  # B,N,C
        m = self.mlp(tokens)    # B,N,H
        return m.permute(0,2,1).contiguous()  # B,H,N

class MaskedSelfAttention(nn.Module):
    def __init__(self, dim, n_heads=4, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.dim = dim
        self.head_dim = dim // n_heads
        assert dim % n_heads == 0
        self.qkv = nn.Linear(dim, dim*3, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)
    def forward(self, x, softmask):  # x: B,N,C ; softmask: B,H,N
        B,N,C = x.shape
        qkv = self.qkv(x).reshape(B,N,3,self.n_heads,self.head_dim).permute(2,0,3,1,4)
        q,k,v = qkv[0], qkv[1], qkv[2]  # B,H,N,D
        attn = (q @ k.transpose(-2,-1)) / (self.head_dim ** 0.5)  # B,H,N,N
        key_bias = torch.log(softmask.unsqueeze(-2) + 1e-6)  # B,H,1,N
        attn = attn + key_bias
        attn = attn.softmax(-1)
        attn = self.attn_drop(attn)
        out = (attn @ v).transpose(1,2).reshape(B,N,C)
        out = self.proj_drop(self.proj(out))
        return out

class MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, int(dim*mlp_ratio))
        self.act = nn.GELU()
        self.fc2 = nn.Linear(int(dim*mlp_ratio), dim)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.fc2(x))
        return x

class MaskedTransformerBlock(nn.Module):
    def __init__(self, dim, n_heads=4, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MaskedSelfAttention(dim, n_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio, drop)
    def forward(self, x, softmask):
        x = x + self.attn(self.norm1(x), softmask)
        x = x + self.mlp(self.norm2(x))
        return x

class AdaptiveMaskedTransformer(nn.Module):
    def __init__(self, in_ch, dim, patch_size=8, depth=2, n_heads=4):
        super().__init__()
        self.pe = PatchEmbed(in_ch, dim, patch_size)
        self.mask_gen = SoftMaskGenerator(dim, hidden=dim//2, n_heads=n_heads)
        self.blocks = nn.ModuleList([MaskedTransformerBlock(dim, n_heads) for _ in range(depth)])
    def forward(self, x):
        tokens, (H,W) = self.pe(x)  # B,N,C
        softmask = self.mask_gen(tokens)  # B,H,N
        for blk in self.blocks:
            tokens = blk(tokens, softmask)
        feat = tokens.transpose(1,2).reshape(x.size(0), tokens.size(-1), H, W)
        return feat


Overwriting braintumnet/src/braintumnet/models/masked_transformer.py


In [38]:
%%writefile braintumnet/src/braintumnet/models/seg_unet.py
import torch, torch.nn as nn
from .cbam import CBAM
from .masked_transformer import AdaptiveMaskedTransformer

def conv_bn_relu(in_ch, out_ch, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(conv_bn_relu(in_ch, out_ch), conv_bn_relu(out_ch, out_ch))
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        x = self.block(x)
        x_down = self.pool(x)
        return x, x_down

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.cbam = CBAM(out_ch + out_ch)
        self.block = nn.Sequential(conv_bn_relu(out_ch + out_ch, out_ch), conv_bn_relu(out_ch, out_ch))
    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, self.cbam(skip)], dim=1)
        x = self.block(x)
        return x

class SegUNetMasked(nn.Module):
    def __init__(self, in_ch=1, base=32, dim=256, patch=8, depth=2, n_heads=4):
        super().__init__()
        self.e1 = EncoderBlock(in_ch, base)
        self.e2 = EncoderBlock(base, base*2)
        self.e3 = EncoderBlock(base*2, base*4)
        self.e4 = EncoderBlock(base*4, base*8)
        self.bottleneck_conv = conv_bn_relu(base*8, dim, k=1, s=1, p=0)
        self.amt = AdaptiveMaskedTransformer(in_ch=dim, dim=dim, patch_size=patch, depth=depth, n_heads=n_heads)
        self.bottleneck_out = conv_bn_relu(dim, base*16, k=1, s=1, p=0)
        self.d4 = DecoderBlock(base*16, base*8)
        self.d3 = DecoderBlock(base*8, base*4)
        self.d2 = DecoderBlock(base*4, base*2)
        self.d1 = DecoderBlock(base*2, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1, x1 = self.e1(x)
        s2, x2 = self.e2(x1)
        s3, x3 = self.e3(x2)
        s4, x4 = self.e4(x3)
        b = self.bottleneck_conv(x4)
        b = self.amt(b)
        b = self.bottleneck_out(b)
        x = self.d4(b, s4)
        x = self.d3(x, s3)
        x = self.d2(x, s2)
        x = self.d1(x, s1)
        seg = self.head(x)
        return seg


Overwriting braintumnet/src/braintumnet/models/seg_unet.py


In [39]:
%%writefile braintumnet/src/braintumnet/models/t_inception.py
import torch.nn as nn
import torch

class InceptionBranch(nn.Module):
    def __init__(self, in_ch, out_ch, k=(3,3)):
        super().__init__()
        if k==(1,1):
            self.op = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        elif k==(1,3):
            self.op = nn.Conv2d(in_ch, out_ch, (1,3), padding=(0,1), bias=False)
        elif k==(3,1):
            self.op = nn.Conv2d(in_ch, out_ch, (3,1), padding=(1,0), bias=False)
        else:
            self.op = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.op(x)))

class TInceptionBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        c = out_ch // 4
        self.b1 = InceptionBranch(in_ch, c, (1,1))
        self.b2 = InceptionBranch(in_ch, c, (3,3))
        self.b3 = InceptionBranch(in_ch, c, (1,3))
        self.b4 = InceptionBranch(in_ch, c, (3,1))
        self.fuse = nn.Conv2d(c*4, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        x = torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)
        return self.act(self.bn(self.fuse(x)))

class TInceptionNet(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.b1 = TInceptionBlock(64, 128)
        self.b2 = TInceptionBlock(128, 256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x)
        x = self.b2(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


Overwriting braintumnet/src/braintumnet/models/t_inception.py


In [40]:
%%writefile braintumnet/src/braintumnet/models/braintumnet.py
import torch, torch.nn as nn
from .seg_unet import SegUNetMasked
from .t_inception import TInceptionNet

class BrainTumNet(nn.Module):
    def __init__(self, in_ch=1, num_cls=2, base=32, dim=256, patch=8, depth=2, n_heads=4, roi_stop_grad=True):
        super().__init__()
        self.seg = SegUNetMasked(in_ch=in_ch, base=base, dim=dim, patch=patch, depth=depth, n_heads=n_heads)
        self.roi_stop_grad = roi_stop_grad
        # classifier consumes ROI gated image (1-ch). If in_ch>1, we can reduce via 1x1 conv or mean.
        self.reduce = nn.Conv2d(in_ch, 1, 1, bias=False) if in_ch>1 else nn.Identity()
        self.cls_backbone = TInceptionNet(in_ch=1, num_classes=num_cls)

    def forward(self, x):
        seg_logits = self.seg(x)  # B,1,H,W
        seg_prob = torch.sigmoid(seg_logits)
        roi_input = self.reduce(x)
        if self.roi_stop_grad:
            roi = roi_input * seg_prob.detach()
        else:
            roi = roi_input * seg_prob
        cls_logits = self.cls_backbone(roi)
        return seg_logits, cls_logits


Overwriting braintumnet/src/braintumnet/models/braintumnet.py


In [41]:
%%writefile braintumnet/src/braintumnet/losses/base.py
import torch, torch.nn as nn, torch.nn.functional as F

def dice_loss_with_logits(logits, target, eps=1e-6):
    pred = torch.sigmoid(logits)
    num = 2 * (pred * target).sum(dim=(2,3))
    den = (pred.pow(2).sum(dim=(2,3)) + target.pow(2).sum(dim=(2,3))) + eps
    dice = 1 - (num + eps) / den
    return dice.mean()

class DiceCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, seg_logits, seg_mask):
        return dice_loss_with_logits(seg_logits, seg_mask) + self.bce(seg_logits, seg_mask)

class MultiTaskLoss(nn.Module):
    def __init__(self, seg_w=1.0, cls_w=0.7):
        super().__init__()
        self.seg_w = seg_w
        self.cls_w = cls_w
        self.seg_loss = DiceCELoss()
        self.cls_loss = nn.CrossEntropyLoss()
    def forward(self, seg_logits, seg_mask, cls_logits, cls_label):
        l_seg = self.seg_loss(seg_logits, seg_mask)
        l_cls = self.cls_loss(cls_logits, cls_label)
        return self.seg_w * l_seg + self.cls_w * l_cls, l_seg.detach(), l_cls.detach()


Overwriting braintumnet/src/braintumnet/losses/base.py


In [42]:
%%writefile braintumnet/src/braintumnet/metrics/base.py
import torch, numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from typing import Tuple

def binarize(logits: torch.Tensor, thr: float=0.5) -> torch.Tensor:
    return (torch.sigmoid(logits) > thr).float()

def iou_score(logits: torch.Tensor, target: torch.Tensor, eps=1e-6) -> float:
    pred = binarize(logits)
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) - inter + eps
    return ((inter + eps) / union).mean().item()

def dice_score(logits: torch.Tensor, target: torch.Tensor, eps=1e-6) -> float:
    pred = binarize(logits)
    num = 2 * (pred * target).sum(dim=(2,3))
    den = pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) + eps
    return (num/den).mean().item()

def hd95_score(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    try:
        from scipy.spatial.distance import cdist
        pred_pts = np.argwhere(pred_mask > 0)
        gt_pts = np.argwhere(gt_mask > 0)
        if len(pred_pts)==0 or len(gt_pts)==0:
            return float("inf")
        D = cdist(pred_pts, gt_pts)
        return np.percentile(np.hstack([D.min(axis=1), D.min(axis=0)]), 95)
    except Exception:
        return float("nan")

def cls_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> Tuple[float,float,float]:
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    auc = float("nan")
    try:
        ncls = y_prob.shape[1]
        if ncls == 2:
            auc = roc_auc_score(y_true, y_prob[:,1])
        else:
            auc = roc_auc_score(y_true, y_prob, multi_class="ovr")
    except Exception:
        pass
    return acc, f1, auc


Overwriting braintumnet/src/braintumnet/metrics/base.py


In [43]:
%%writefile braintumnet/src/braintumnet/engine/trainer.py
import os, math, torch, time
from torch.utils.data import DataLoader
from typing import Dict
from ..models.braintumnet import BrainTumNet
from ..data.brats2020_dataset import SliceDataset
from ..losses import MultiTaskLoss
from ..metrics import iou_score, dice_score
from ..utils.io import ensure_dir, save_ckpt

def _cosine_lr(optimizer, base_lr, t, T):
    lr = 0.5 * base_lr * (1 + math.cos(math.pi * t / T))
    for pg in optimizer.param_groups: pg["lr"] = lr

def build_dataloaders(cfg: Dict, fold: int):
    proc = cfg["data"]["proc_root"]
    img_size = cfg["data"]["img_size"]
    train_list = os.path.join(proc, f"split_train_fold{fold}.txt")
    val_list   = os.path.join(proc, f"split_val_fold{fold}.txt")
    train_ds = SliceDataset(proc, train_list, img_size, cfg["augment"]["rotate_deg"],
                            cfg["augment"]["hflip_p"], cfg["augment"]["vflip_p"], True, cfg["model"]["in_channels"])
    val_ds   = SliceDataset(proc, val_list, img_size, 0,0,0, False, cfg["model"]["in_channels"])
    train_loader = DataLoader(train_ds, batch_size=cfg["train"]["batch_size"], shuffle=True, num_workers=cfg["train"]["workers"])
    val_loader   = DataLoader(val_ds, batch_size=cfg["train"]["batch_size"], shuffle=False, num_workers=cfg["train"]["workers"])
    return train_loader, val_loader

def build_model(cfg: Dict):
    mcfg = cfg["model"]
    return BrainTumNet(in_ch=mcfg["in_channels"], num_cls=mcfg["num_classes_cls"], base=mcfg["base"],
                       dim=mcfg["dim"], patch=mcfg["patch_size"], depth=mcfg["depth"], n_heads=mcfg["n_heads"],
                       roi_stop_grad=mcfg["roi_stop_grad"])

def train_one_fold(cfg: Dict, fold: int):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_loader, val_loader = build_dataloaders(cfg, fold)
    model = build_model(cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["train"]["lr"], weight_decay=cfg["train"]["weight_decay"])
    crit = MultiTaskLoss(cfg["train"]["seg_loss_weight"], cfg["train"]["cls_loss_weight"])
    scaler = torch.cuda.amp.GradScaler(enabled=cfg["train"].get("amp", False))

    total_steps = cfg["train"]["epochs"] * max(1, len(train_loader))
    step = 0
    best_iou = -1.0

    for epoch in range(cfg["train"]["epochs"]):
        model.train()
        for batch in train_loader:
            img = batch["image"].to(device)
            msk = batch["mask"].to(device)
            lab = batch["label"].to(device)
            with torch.cuda.amp.autocast(enabled=cfg["train"].get("amp", False)):
                seg, cls = model(img)
                loss, _, _ = crit(seg, msk, cls, lab)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if cfg["train"]["scheduler"] == "cosine":
                _cosine_lr(opt, cfg["train"]["lr"], step, total_steps)
            step += 1

        # validation
        model.eval()
        iou_m, dice_m, acc_m, n = 0.0, 0.0, 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                img = batch["image"].to(device)
                msk = batch["mask"].to(device)
                lab = batch["label"].to(device)
                seg, cls = model(img)
                iou_m += iou_score(seg, msk)
                dice_m += dice_score(seg, msk)
                acc_m += (cls.argmax(1)==lab).float().mean().item()
                n += 1
        iou_m /= n; dice_m /= n; acc_m /= n
        print(f"[Fold {fold}] Epoch {epoch+1}/{cfg['train']['epochs']} | IoU {iou_m:.4f} | Dice {dice_m:.4f} | ClsAcc {acc_m:.4f}")

        if iou_m > best_iou:
            best_iou = iou_m
            ckpt_dir = cfg["logging"]["save_dir"]
            ensure_dir(ckpt_dir)
            save_ckpt(model, os.path.join(ckpt_dir, f"braintumnet_best_fold{fold}.pth"))

    return best_iou


Overwriting braintumnet/src/braintumnet/engine/trainer.py


In [44]:
%%writefile braintumnet/src/braintumnet/engine/evaluator.py
import os, torch, numpy as np
from torch.utils.data import DataLoader
from typing import Dict
from ..models.braintumnet import BrainTumNet
from ..data.brats2020_dataset import SliceDataset
from ..metrics import cls_metrics
from ..utils.io import load_ckpt

def evaluate(cfg: Dict, fold: int, ckpt_path: str):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    proc = cfg["data"]["proc_root"]
    val_list = os.path.join(proc, f"split_val_fold{fold}.txt")
    ds = SliceDataset(proc, val_list, cfg["data"]["img_size"], train=False, in_channels=cfg["model"]["in_channels"])
    dl = DataLoader(ds, batch_size=cfg["train"]["batch_size"], shuffle=False, num_workers=cfg["train"]["workers"])
    model = BrainTumNet(in_ch=cfg["model"]["in_channels"], num_cls=cfg["model"]["num_classes_cls"],
                        base=cfg["model"]["base"], dim=cfg["model"]["dim"], patch=cfg["model"]["patch_size"],
                        depth=cfg["model"]["depth"], n_heads=cfg["model"]["n_heads"],
                        roi_stop_grad=cfg["model"]["roi_stop_grad"]).to(device)
    load_ckpt(model, ckpt_path, map_location=device)
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    import torch.nn.functional as F
    with torch.no_grad():
        for batch in dl:
            img = batch["image"].to(device)
            lab = batch["label"].cpu().numpy()
            seg, cls = model(img)
            prob = F.softmax(cls, dim=1).cpu().numpy()
            y_true.extend(lab.tolist())
            y_pred.extend(prob.argmax(1).tolist())
            y_prob.extend(prob.tolist())
    y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)
    acc, f1, auc = cls_metrics(y_true, y_pred, y_prob)
    print(f"[Fold {fold}] ACC {acc:.4f} | F1 {f1:.4f} | AUC {auc:.4f}")


Overwriting braintumnet/src/braintumnet/engine/evaluator.py


In [45]:
%%writefile braintumnet/scripts/prepare_brats2020.py
import os, argparse
from pathlib import Path
import sys
ROOT = Path(__file__).resolve().parents[1]  # braintumnet/
sys.path.append(str(ROOT / "src"))

from braintumnet.data.preprocessing import process_brats2020, make_folds

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", required=True, help="Path to BraTS2020 TrainingData root (contains HGG/LGG)")
    ap.add_argument("--out", required=True, help="Output processed dir (e.g., braintumnet/data/processed)")
    ap.add_argument("--modality", default="t1ce", choices=["t1","t1ce","t2","flair"])
    ap.add_argument("--img_size", type=int, default=256)
    ap.add_argument("--slices_per_case", type=int, default=20)
    ap.add_argument("--tumor_slice_ratio", type=float, default=0.7)
    ap.add_argument("--num_folds", type=int, default=5)
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    process_brats2020(args.raw, args.out, args.modality, args.img_size, args.slices_per_case, args.tumor_slice_ratio)
    make_folds(args.out, args.num_folds)

if __name__ == "__main__":
    main()


Overwriting braintumnet/scripts/prepare_brats2020.py


In [46]:
%%writefile braintumnet/scripts/train.py
import os, argparse, sys
from pathlib import Path
ROOT = Path(__file__).resolve().parents[1]  # braintumnet/
sys.path.append(str(ROOT / "src"))

from braintumnet.utils.io import load_yaml
from braintumnet.utils.seed import set_seed
from braintumnet.engine.trainer import train_one_fold

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", type=str, default=str(ROOT / "configs" / "default.yaml"))
    ap.add_argument("--fold", type=int, default=None, help="0..K-1. If not set, uses cfg.data.fold")
    args = ap.parse_args()

    cfg = load_yaml(args.cfg)
    if args.fold is not None:
        cfg["data"]["fold"] = args.fold

    set_seed(42, deterministic=False)
    best_iou = train_one_fold(cfg, cfg["data"]["fold"])
    print("Best IoU:", best_iou)

if __name__ == "__main__":
    main()


Overwriting braintumnet/scripts/train.py


In [47]:
%%writefile braintumnet/scripts/evaluate.py
import os, argparse, sys
from pathlib import Path
ROOT = Path(__file__).resolve().parents[1]
sys.path.append(str(ROOT / "src"))

from braintumnet.utils.io import load_yaml
from braintumnet.engine.evaluator import evaluate

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", type=str, default=str(ROOT / "configs" / "default.yaml"))
    ap.add_argument("--ckpt", type=str, required=True)
    ap.add_argument("--fold", type=int, default=None)
    args = ap.parse_args()

    cfg = load_yaml(args.cfg)
    if args.fold is not None:
        cfg["data"]["fold"] = args.fold

    evaluate(cfg, cfg["data"]["fold"], args.ckpt)

if __name__ == "__main__":
    main()


Overwriting braintumnet/scripts/evaluate.py


In [48]:
%%writefile braintumnet/scripts/visualize_batch.py
import os, sys, argparse
from pathlib import Path
ROOT = Path(__file__).resolve().parents[1]
sys.path.append(str(ROOT / "src"))

import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from braintumnet.utils.io import load_yaml
from braintumnet.data.brats2020_dataset import SliceDataset

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", type=str, default=str(ROOT / "configs" / "default.yaml"))
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--n", type=int, default=8)
    args = ap.parse_args()
    cfg = load_yaml(args.cfg)

    proc = cfg["data"]["proc_root"]
    split = os.path.join(proc, f"split_train_fold{args.fold}.txt")
    ds = SliceDataset(proc, split, cfg["data"]["img_size"], train=True, in_channels=cfg["model"]["in_channels"])
    dl = DataLoader(ds, batch_size=args.n, shuffle=True)
    batch = next(iter(dl))
    imgs = batch["image"]; msks = batch["mask"]
    n = imgs.size(0)
    cols = 4
    rows = (n*2 + cols - 1)//cols
    plt.figure(figsize=(cols*3, rows*3))
    for i in range(n):
        plt.subplot(rows, cols, i*2+1); plt.imshow(imgs[i,0].numpy(), cmap="gray"); plt.axis("off"); plt.title("img")
        plt.subplot(rows, cols, i*2+2); plt.imshow(msks[i,0].numpy(), cmap="gray"); plt.axis("off"); plt.title("mask")
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()


Overwriting braintumnet/scripts/visualize_batch.py


In [ ]:
python braintumnet/scripts/prepare_brats2020.py 
  --raw E:/thong/code/brain_segmen/brats2020_data/bcs2020/archive/BraTS2020_training_data 
  --out braintumnet/data/processed 
  --modality t1ce --img_size 256 --slices_per_case 20 --tumor_slice_ratio 0.7

Processed slices: 0
Folds written: 5
